# Baseline accuracy of UCSF-PDGM using Logistic Regression

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

## Load and clean dataset

In [ ]:
ds = load_dataset("chehablab/UCSF_PDGM", split="train", keep_in_memory=True)
df = pd.DataFrame(ds, columns=["volume_id", "slice_id", "t1", "t1c", "t2", "is_tumorous", "sex", "age"])
df = df.dropna(subset=["sex"]) # Remove all empty metadata columns
df.sex = (df.sex == "M").astype(np.float32)

## Split dataset into pixel and metadata as x training and test sets and is_tumourous as y

In [ ]:
# Split the dataset
patient_ids = np.asarray(df["volume_id"].unique(), dtype=object) # Splits by patient
train_ids, test_ids = train_test_split(patient_ids, test_size=0.3, random_state=1606009)
train_df = df[df["volume_id"].isin(train_ids).reset_index(drop=True)]
test_df = df[df["volume_id"].isin(test_ids).reset_index(drop=True)]

In [ ]:
# Split into x for metadata and pixels and y sets
metadata = ["slice_id", "sex", "age"]
pixel = ["t1", "t1c", "t2"]

y_train = train_df["is_tumorous"]
y_test = test_df["is_tumorous"]
x_train_meta = train_df[metadata]
x_train_pixel = train_df[pixel]
x_test_meta = test_df[metadata]
x_test_pixel = test_df[pixel]

## Logistic Regression on metadata to predict is_tumourous

In [ ]:
# Metadata logistic regression
meta_scaler = StandardScaler()
x_train_meta_scaled = meta_scaler.fit_transform(x_train_meta)
x_test_meta_scaled = meta_scaler.fit_transform(x_test_meta)

logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train_meta_scaled, y_train)
meta_pred = logreg.predict(x_test_meta_scaled)
meta_proba = logreg.predict_proba(x_test_meta_scaled)

print("Baseline metadata logistic regression")
print(classification_report(y_test, meta_pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, meta_proba):.4f}")

## Logistic Regression on pixels to predict is_tumourous

In [ ]:
# Pixels logistic regression
def flatten_pixels(t1, t1c, t2):
    channels = []
    for arr in (t1, t1c, t2):
        arr = np.array(arr, dtype=np.float32)
        img = Image.fromarray(arr).resize((128, 128), Image.Resampling.BILINEAR)
        ch = np.array(img, dtype=np.float32)
        lo, hi = ch.min(), ch.max()
        ch = (ch - lo)/ (hi - lo + 1e-8)
        channels.append(ch)
    return np.concatenate(channels) # (128, 128, 3)

print("Flattening images to 128*128*3 (T1, T1c, T2) for pixel baseline...")
x_train_pixel_flat = np.stack([flatten_pixels(r["t1"], r["t1c"], r["t2"]) for _, r in x_train_pixel.iterrows()])
x_test_pixel_flat = np.stack([flatten_pixels(r["t1"], r["t1c"], r["t2"]) for _, r in x_test_pixel.iterrows()])
logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train_meta_scaled, y_train)
pixel_pred = logreg.predict(x_train_pixel_flat)
pixel_proba = logreg.predict_proba(x_test_pixel_flat)

print("Baseline pixel logistic regression")
print(classification_report(y_test, pixel_pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, pixel_proba):.4f}")

## Logistic Regression on metadata and pixels to predict is_tumourous

In [ ]:
# Combined logistic regression
x_train = np.hstack([x_train_meta_scaled, x_train_pixel_flat])
x_test = np.hstack([x_test_meta_scaled, x_test_pixel_flat])
logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train, y_train)
pred = logreg.predict(x_train)
proba = logreg.predict_proba(x_train)

print("Baseline combined pixel + metadata logistic regression")
print(classification_report(y_test, pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")